<a href="https://colab.research.google.com/github/aditya450134/tugas-p2-telco-churn/blob/main/praproses_telco_churn_py.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# Tugas P2 - Pra-pemrosesan data
# Dataset : Telco Customer Churn (Kaggle)
# Kasus   : klasifikasi pelanggan berhenti berlangganan atau tidak (Churn)
# Metode  : Naive Bayes Gaussian (NIM ganjil)

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, classification_report

pd.set_option("display.width", 90)
pd.set_option("display.max_columns", 50)

# baca data
# link asli: https://www.kaggle.com/datasets/blastchar/telco-customer-churn
# yang dipakai di sini salinan datanya di github (IBM)
url = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
df = pd.read_csv(url)

print("Ukuran data:", df.shape)
print(df.head())

# ---------- info dan tipe kolom ----------
print("\nInfo kolom:")
df.info()

# TotalCharges harusnya angka tapi kebaca teks, dicek dulu kenapa
kosong = df[df["TotalCharges"].str.strip() == ""]
print("\nTotalCharges yang kosong:", len(kosong), "baris")
print(kosong[["customerID", "tenure", "TotalCharges"]].head())

# diubah ke angka, yang kosong otomatis jadi NaN
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
print("Tipe TotalCharges sekarang:", df["TotalCharges"].dtype)

# ---------- cek kategori ----------
kolom_teks = df.select_dtypes(exclude="number").columns.drop("customerID")

print("\nIsi tiap kolom kategori:")
for k in kolom_teks:
    df[k] = df[k].str.strip()
    print(k, ":", sorted(df[k].unique()))

# cek huruf besar kecil
for k in kolom_teks:
    if df[k].nunique() != df[k].str.lower().nunique():
        print("kapitalisasi tidak konsisten di", k)
print("cek kapitalisasi selesai")

# "No internet service" dan "No phone service" artinya sama dengan "No"
# (sudah terwakili kolom InternetService dan PhoneService)
df = df.replace({"No internet service": "No", "No phone service": "No"})
print("OnlineSecurity sekarang:", sorted(df["OnlineSecurity"].unique()))

# ---------- missing value ----------
print("\nMissing value:")
print(df.isnull().sum()[df.isnull().sum() > 0])

# ---------- duplikasi ----------
print("\nBaris duplikat:", df.duplicated().sum())
print("customerID duplikat:", df["customerID"].duplicated().sum())
# baris yang isinya sama kalau customerID diabaikan
# (customerID-nya beda, jadi dianggap pelanggan berbeda dan tidak dihapus)
print("Baris dengan isi fitur sama (tanpa customerID):",
      df.drop(columns="customerID").duplicated().sum())
df = df.drop_duplicates().reset_index(drop=True)

# ---------- fitur tidak relevan dan leakage ----------
# customerID cuma nomor pelanggan, jadi dibuang
df = df.drop(columns=["customerID"])

# target dibuat angka dulu supaya bisa dihitung korelasinya
df["target"] = (df["Churn"] == "Yes").astype(int)

print("\nKorelasi dengan target:")
print(df[["SeniorCitizen", "tenure", "MonthlyCharges", "TotalCharges", "target"]].corr()["target"].round(3))
print("Korelasi tenure dan TotalCharges:", round(df["tenure"].corr(df["TotalCharges"]), 3))

df = df.drop(columns=["Churn"])
print("Jumlah fitur:", df.shape[1] - 1)

# ---------- kelas tidak seimbang ----------
print("\nJumlah tiap kelas:")
print(df["target"].value_counts())
print((df["target"].value_counts(normalize=True) * 100).round(2))

# ---------- bagi data 80:20 ----------
# dibagi sebelum isi missing value, outlier, dan scaling
# supaya perhitungannya cuma dari data latih
X = df.drop(columns=["target"])
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
print("\nData latih:", X_train.shape, " Data uji:", X_test.shape)

# ---------- isi missing value pakai median data latih ----------
median_total = X_train["TotalCharges"].median()
X_train["TotalCharges"] = X_train["TotalCharges"].fillna(median_total)
X_test["TotalCharges"] = X_test["TotalCharges"].fillna(median_total)

print("\nTotalCharges diisi dengan median:", median_total)
print("Sisa missing value (latih):", X_train.isnull().sum().sum())
print("Sisa missing value (uji):", X_test.isnull().sum().sum())

# ---------- outlier pakai IQR ----------
print("\nOutlier (IQR):")
for k in ["tenure", "MonthlyCharges", "TotalCharges"]:
    q1 = X_train[k].quantile(0.25)
    q3 = X_train[k].quantile(0.75)
    iqr = q3 - q1
    bawah = q1 - 1.5 * iqr
    atas = q3 + 1.5 * iqr
    jumlah = ((X_train[k] < bawah) | (X_train[k] > atas)).sum()
    print(k, "-> Q1:", round(q1, 2), "Q3:", round(q3, 2),
          "batas:", round(bawah, 2), "sampai", round(atas, 2),
          "| jumlah outlier:", jumlah)
    # kalau ada, nilainya dibatasi ke batas atas/bawah (winsorizing)
    X_train[k] = X_train[k].clip(bawah, atas)
    X_test[k] = X_test[k].clip(bawah, atas)

# ---------- ubah kategori jadi angka ----------
# kolom Yes/No jadi 1/0
kolom_yn = ["Partner", "Dependents", "PhoneService", "MultipleLines",
            "OnlineSecurity", "OnlineBackup", "DeviceProtection",
            "TechSupport", "StreamingTV", "StreamingMovies", "PaperlessBilling"]
for k in kolom_yn:
    X_train[k] = (X_train[k] == "Yes").astype(int)
    X_test[k] = (X_test[k] == "Yes").astype(int)

# gender: Male = 1, Female = 0
X_train["gender"] = (X_train["gender"] == "Male").astype(int)
X_test["gender"] = (X_test["gender"] == "Male").astype(int)

# Contract ada urutannya (ordinal)
urutan = {"Month-to-month": 1, "One year": 2, "Two year": 3}
X_train["Contract"] = X_train["Contract"].map(urutan)
X_test["Contract"] = X_test["Contract"].map(urutan)

# InternetService dan PaymentMethod tidak ada urutan, pakai one hot
X_train = pd.get_dummies(X_train, columns=["InternetService", "PaymentMethod"], dtype=int)
X_test = pd.get_dummies(X_test, columns=["InternetService", "PaymentMethod"], dtype=int)

# kolom data uji disamakan dengan data latih
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

# ---------- standarisasi dan normalisasi ----------
# standarisasi untuk kolom numerik (menyamakan skala antar fitur)
kolom_std = ["tenure", "MonthlyCharges", "TotalCharges"]
std = StandardScaler()
X_train[kolom_std] = std.fit_transform(X_train[kolom_std])
X_test[kolom_std] = std.transform(X_test[kolom_std])

# normalisasi min-max untuk Contract (1-3 jadi 0-1)
mm = MinMaxScaler()
X_train[["Contract"]] = mm.fit_transform(X_train[["Contract"]])
X_test[["Contract"]] = mm.transform(X_test[["Contract"]])

# ---------- hasil pemrosesan ----------
# data latih dan uji digabung lagi, diurutkan seperti data asli
latih = X_train.copy()
latih["target"] = y_train.values
uji = X_test.copy()
uji["target"] = y_test.values
hasil = pd.concat([latih, uji]).sort_index()

print("\nHasil pemrosesan data:", hasil.shape)

# ditampilkan transpose supaya muat
print("\n5 data teratas:")
print(hasil.head(5).T.round(2))
print("\n5 data terbawah:")
print(hasil.tail(5).T.round(2))

# ---------- klasifikasi Naive Bayes Gaussian ----------
model = GaussianNB()
model.fit(X_train, y_train)
prediksi = model.predict(X_test)

print("\nAkurasi:", round(accuracy_score(y_test, prediksi), 4))
print(classification_report(y_test, prediksi, target_names=["No", "Yes"]))

Ukuran data: (7043, 21)
   customerID  gender  SeniorCitizen Partner Dependents  tenure PhoneService  \
0  7590-VHVEG  Female              0     Yes         No       1           No   
1  5575-GNVDE    Male              0      No         No      34          Yes   
2  3668-QPYBK    Male              0      No         No       2          Yes   
3  7795-CFOCW    Male              0      No         No      45           No   
4  9237-HQITU  Female              0      No         No       2          Yes   

      MultipleLines InternetService OnlineSecurity OnlineBackup DeviceProtection  \
0  No phone service             DSL             No          Yes               No   
1                No             DSL            Yes           No              Yes   
2                No             DSL            Yes          Yes               No   
3  No phone service             DSL            Yes           No              Yes   
4                No     Fiber optic             No           No            